# Biomedical Regression: Predicting Drug Response from Gene Expression
## Assessment 2 — Statistical Learning


## Phase 1: Exploratory Data Analysis


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Load both datasets
train = pd.read_csv('train1.csv')
test = pd.read_csv('test_public.csv')
features = [f'X{i}' for i in range(1, 41)]

print(f"Training set: {train.shape[0]} samples, {train.shape[1]} columns")
print(f"Test set (test_public.csv): {test.shape[0]} samples, {test.shape[1]} columns")
print(f"Test columns: {list(test.columns[:5])}... (obs_id + 40 features, no y)")
print(f"\nLaboratories in training: {train['group_id'].nunique()}")
print(f"Samples per laboratory: {train.groupby('group_id').size().unique()[0]}")
print(f"\nResponse y: mean={train['y'].mean():.2f}, std={train['y'].std():.2f}, range=[{train['y'].min():.2f}, {train['y'].max():.2f}]")
print(f"Missing values — train: {train.isnull().sum().sum()}, test: {test.isnull().sum().sum()}")


In [ ]:
# Skewness check
skew = train[features].skew()
print("Max |skewness|:", skew.abs().max().round(3), "- no transformation needed")

# Correlation with y
corr_y = train[features + ['y']].corr()['y'].drop('y').sort_values(key=abs, ascending=False)
print("\nTop feature correlations with y:")
print(corr_y.head(10))

# Batch effects
group_means = train.groupby('group_id')['y'].mean()
print(f"\nBatch effects: lab mean range [{group_means.min():.1f}, {group_means.max():.1f}]")
print(f"Std of lab means: {group_means.std():.2f} vs overall std: {train['y'].std():.2f}")
print("=> Batch effects are dominant — must use GroupKFold validation")


In [ ]:
# EDA Figure
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Response distribution
axes[0,0].hist(train['y'], bins=40, edgecolor='black', alpha=0.7, color='steelblue')
axes[0,0].axvline(train['y'].mean(), color='red', ls='--', label=f"Mean={train['y'].mean():.1f}")
axes[0,0].set_xlabel('y'); axes[0,0].set_title('Distribution of y'); axes[0,0].legend()

# Batch effects boxplot
group_order = train.groupby('group_id')['y'].mean().sort_values().index
axes[0,1].boxplot([train[train['group_id']==g]['y'].values for g in group_order],
                  patch_artist=True, showfliers=False)
axes[0,1].set_title('y by Laboratory (sorted by mean)'); axes[0,1].set_xticks([])
axes[0,1].set_xlabel('Laboratory'); axes[0,1].set_ylabel('y')

# PCA
scaler = StandardScaler()
X_sc = scaler.fit_transform(train[features])
pca2d = PCA(n_components=2)
pcs = pca2d.fit_transform(X_sc)
axes[1,0].scatter(pcs[:,0], pcs[:,1], c=train['group_id'], cmap='tab20', alpha=0.4, s=10)
axes[1,0].set_xlabel(f'PC1 ({pca2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[1,0].set_ylabel(f'PC2 ({pca2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[1,0].set_title('PCA coloured by Laboratory')

# Correlations with y
corr_y_all = train[features+['y']].corr()['y'].drop('y').sort_values(ascending=False)
colors_bar = ['steelblue' if v>0 else 'salmon' for v in corr_y_all]
axes[1,1].barh(corr_y_all.index, corr_y_all.values, color=colors_bar)
axes[1,1].set_title('Feature Correlations with y'); axes[1,1].axvline(0, color='black', lw=0.5)
plt.tight_layout(); plt.show()


In [ ]:
# Correlation heatmap of top features
top = ['X5','X6','X29','X19','X36','X13','X37','X11','X32','X38','y']
fig, ax = plt.subplots(figsize=(8,7))
mask = np.triu(np.ones((len(top),len(top)),dtype=bool), k=1)
sns.heatmap(train[top].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, mask=mask, square=True, ax=ax)
ax.set_title('Correlation Matrix: Top Predictors + y')
plt.tight_layout(); plt.show()


In [ ]:
# Scree plot
pca_full = PCA().fit(StandardScaler().fit_transform(train[features]))
cum_pve = np.cumsum(pca_full.explained_variance_ratio_)
n90 = np.argmax(cum_pve >= 0.9) + 1
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(range(1,41), pca_full.explained_variance_ratio_, alpha=0.6, color='steelblue', label='Individual')
ax.plot(range(1,41), cum_pve, 'ro-', markersize=4, label='Cumulative')
ax.axhline(0.9, ls='--', color='grey'); ax.set_xlabel('PC'); ax.set_ylabel('PVE')
ax.set_title(f'Scree Plot ({n90} PCs for 90% variance)'); ax.legend()
plt.tight_layout(); plt.show()


## Phase 2: Model Development

Two regularised linear models are compared using GroupKFold cross-validation, plus PCR as a supplementary comparison.


In [ ]:
X_train = train[features].values
y_train = train['y'].values
groups = train['group_id'].values
X_test = test[features].values  # test_public.csv features

gkf = GroupKFold(n_splits=10)
print("Validation: GroupKFold (k=10) — each fold holds out 4 entire laboratories")
print("This mimics the test scenario: predicting on labs NOT seen during training\n")

# === Model 1: Lasso (L1 Regularisation) ===
lasso_alphas = np.logspace(-4, 1, 50)
lasso_grid = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('lasso', Lasso(max_iter=10000))]),
    param_grid={'lasso__alpha': lasso_alphas},
    cv=gkf, scoring='neg_mean_squared_error', n_jobs=-1, return_train_score=True
)
lasso_grid.fit(X_train, y_train, groups=groups)
print(f"Lasso: best alpha={lasso_grid.best_params_['lasso__alpha']:.4f}, "
      f"CV MSE={-lasso_grid.best_score_:.4f} +/- {lasso_grid.cv_results_['std_test_score'][lasso_grid.best_index_]:.4f}")

# === Model 2: Ridge (L2 Regularisation) ===
ridge_alphas = np.logspace(-2, 4, 50)
ridge_grid = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('ridge', Ridge())]),
    param_grid={'ridge__alpha': ridge_alphas},
    cv=gkf, scoring='neg_mean_squared_error', n_jobs=-1, return_train_score=True
)
ridge_grid.fit(X_train, y_train, groups=groups)
print(f"Ridge: best alpha={ridge_grid.best_params_['ridge__alpha']:.2f}, "
      f"CV MSE={-ridge_grid.best_score_:.4f} +/- {ridge_grid.cv_results_['std_test_score'][ridge_grid.best_index_]:.4f}")

# === PCR (supplementary) ===
pcr_grid = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('pca', PCA()), ('lr', LinearRegression())]),
    param_grid={'pca__n_components': list(range(1,41))},
    cv=gkf, scoring='neg_mean_squared_error', n_jobs=-1, return_train_score=True
)
pcr_grid.fit(X_train, y_train, groups=groups)
print(f"PCR:   best M={pcr_grid.best_params_['pca__n_components']}, "
      f"CV MSE={-pcr_grid.best_score_:.4f} +/- {pcr_grid.cv_results_['std_test_score'][pcr_grid.best_index_]:.4f}")

print(f"\n=> Lasso achieves the lowest CV MSE. Selected as final model.")


In [ ]:
# Lasso coefficients — which genes are selected?
lasso_best = Pipeline([('scaler', StandardScaler()),
                       ('lasso', Lasso(alpha=lasso_grid.best_params_['lasso__alpha'], max_iter=10000))])
lasso_best.fit(X_train, y_train)
coefs = pd.Series(lasso_best.named_steps['lasso'].coef_, index=features)
print(f"Lasso selects {(coefs!=0).sum()}/40 features:")
print(coefs[coefs!=0].sort_values(key=abs, ascending=False))

# Ridge coefficients for comparison
ridge_best = Pipeline([('scaler', StandardScaler()),
                       ('ridge', Ridge(alpha=ridge_grid.best_params_['ridge__alpha']))])
ridge_best.fit(X_train, y_train)
ridge_coefs = pd.Series(ridge_best.named_steps['ridge'].coef_, index=features)
print(f"\nRidge top 10 coefficients (all 40 non-zero):")
print(ridge_coefs.reindex(ridge_coefs.abs().sort_values(ascending=False).index).head(10))


## Phase 3: Validation Plots and Model Selection


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Lasso CV path
la = np.array(lasso_alphas)
lm = -lasso_grid.cv_results_['mean_test_score']
ls_ = lasso_grid.cv_results_['std_test_score']
axes[0,0].semilogx(la, lm, 'b-o', ms=3, label='CV MSE')
axes[0,0].fill_between(la, lm-ls_, lm+ls_, alpha=0.2)
axes[0,0].semilogx(la, -lasso_grid.cv_results_['mean_train_score'], 'g--', alpha=0.6, label='Train MSE')
axes[0,0].axvline(lasso_grid.best_params_['lasso__alpha'], color='r', ls='--', label='Best alpha')
axes[0,0].set_xlabel('alpha'); axes[0,0].set_ylabel('MSE'); axes[0,0].set_title('Lasso CV'); axes[0,0].legend(fontsize=8)

# Ridge CV path
ra = np.array(ridge_alphas)
rm = -ridge_grid.cv_results_['mean_test_score']
rs = ridge_grid.cv_results_['std_test_score']
axes[0,1].semilogx(ra, rm, 'b-o', ms=3, label='CV MSE')
axes[0,1].fill_between(ra, rm-rs, rm+rs, alpha=0.2)
axes[0,1].semilogx(ra, -ridge_grid.cv_results_['mean_train_score'], 'g--', alpha=0.6, label='Train MSE')
axes[0,1].axvline(ridge_grid.best_params_['ridge__alpha'], color='r', ls='--', label='Best alpha')
axes[0,1].set_xlabel('alpha'); axes[0,1].set_ylabel('MSE'); axes[0,1].set_title('Ridge CV'); axes[0,1].legend(fontsize=8)

# PCR components
nc = np.arange(1,41)
pm = -pcr_grid.cv_results_['mean_test_score']
ps = pcr_grid.cv_results_['std_test_score']
axes[1,0].plot(nc, pm, 'b-o', ms=3, label='CV MSE')
axes[1,0].fill_between(nc, pm-ps, pm+ps, alpha=0.2)
axes[1,0].plot(nc, -pcr_grid.cv_results_['mean_train_score'], 'g--', alpha=0.6, label='Train MSE')
axes[1,0].axvline(pcr_grid.best_params_['pca__n_components'], color='r', ls='--', label='Best M')
axes[1,0].set_xlabel('Components'); axes[1,0].set_ylabel('MSE'); axes[1,0].set_title('PCR CV'); axes[1,0].legend(fontsize=8)

# Lasso selected coefficients
nz = coefs[coefs!=0].sort_values()
axes[1,1].barh(nz.index, nz.values, color=['steelblue' if v>0 else 'salmon' for v in nz])
axes[1,1].set_xlabel('Coefficient'); axes[1,1].set_title(f'Lasso: {len(nz)}/40 features selected')
axes[1,1].axvline(0, color='black', lw=0.5)
plt.tight_layout(); plt.show()


In [ ]:
# Coefficient comparison plot
fig, ax = plt.subplots(figsize=(12, 5))
x_pos = np.arange(40)
w = 0.35
ax.bar(x_pos - w/2, coefs.values, w, label='Lasso', color='steelblue', alpha=0.8)
ax.bar(x_pos + w/2, ridge_coefs.values, w, label='Ridge', color='coral', alpha=0.8)
ax.set_xticks(x_pos); ax.set_xticklabels(features, rotation=90, fontsize=7)
ax.set_ylabel('Coefficient (standardised)'); ax.set_title('Lasso vs Ridge Coefficients')
ax.legend(); ax.axhline(0, color='black', lw=0.5)
plt.tight_layout(); plt.show()


## Phase 4: Test Predictions on test_public.csv

Refit the selected Lasso model on ALL training data, then generate predictions for the test set.


In [ ]:
# ============================================================
# FINAL MODEL: Refit on entire training set
# ============================================================
best_alpha = lasso_grid.best_params_['lasso__alpha']
final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=best_alpha, max_iter=10000))
])
final_model.fit(X_train, y_train)

train_pred = final_model.predict(X_train)
train_mse = mean_squared_error(y_train, train_pred)
train_r2 = 1 - train_mse / np.var(y_train)
print(f"Final model: Lasso (alpha={best_alpha:.4f})")
print(f"Training MSE = {train_mse:.4f}, R^2 = {train_r2:.4f}")
print(f"Estimated generalisation error (GroupKFold CV): MSE = {-lasso_grid.best_score_:.4f}")

# ============================================================
# PREDICT ON test_public.csv
# ============================================================
print(f"\n--- Generating predictions on test_public.csv ---")
print(f"Test set loaded: {test.shape[0]} observations, columns: {list(test.columns[:3])}...")

y_hat = final_model.predict(X_test)

predictions = pd.DataFrame({
    'obs_id': test['obs_id'],
    'y_hat': y_hat
})

# Verify format matches requirements
assert list(predictions.columns) == ['obs_id', 'y_hat'], "Column format check failed"
assert len(predictions) == 1000, "Row count check failed"
assert predictions['y_hat'].dtype == float, "Predictions must be continuous"

print(f"\nPrediction summary:")
print(f"  Rows: {len(predictions)}")
print(f"  Columns: {list(predictions.columns)}")
print(f"  y_hat — mean: {predictions['y_hat'].mean():.3f}, std: {predictions['y_hat'].std():.3f}")
print(f"  y_hat — range: [{predictions['y_hat'].min():.3f}, {predictions['y_hat'].max():.3f}]")

# Save predictions CSV
predictions.to_csv('predictions.csv', index=False)
print(f"\nSaved predictions.csv ({len(predictions)} rows)")
print(predictions.head(10))
